# R2.09 TP 3 : Le cryptosystème RSA (SageMath)

**Étudiant :** Mamadou-Bailo BARRY - BUT Informatique 2 (Site d'Arles)  
**Enseignant :** M. Bruno Colombel  
**Environnement :** SageMath


## 1 Calcul de puissances modulaires

Le chiffrement comme le déchiffrement par RSA reposent sur le calcul de puissances modulaires $a^r \pmod m$.
On implémente l'algorithme des carrés répétés vu en cours et en TD (avec les colonnes $r, a, p$) pour effectuer ce calcul de manière efficace.

### Exercice 1 : Fonction expo_mod(a, r, m) en sagemath

In [1]:
def expo_mod(a, r, m):
    p = 1
    while r > 0:
        if r % 2 == 1:
            p = (p * a) % m
            r = (r - 1) // 2
        else:
            r = r // 2
        a = (a * a) % m
    return p

Exemple d'utilisation donné dans le sujet :

In [2]:
expo_mod(13, 5, 8)

5

## 2 Rappels de cours & 3 Mise en œuvre sur un exemple simple

### 3.1 Génération des clés
On prend $p = 47$ et $q = 59$. On vérifie que $p$ et $q$ sont bien premiers avec la méthode `.is_prime()` :

In [3]:
p = 47 ; q = 59
p.is_prime(), q.is_prime()

(True, True)

On peut alors calculer $n = p \cdot q$ et la fonction indicatrice d'Euler $\varphi(n)$ avec `euler_phi(n)` :

In [4]:
n = p * q
phi = euler_phi(n)
n, phi

(2773, 2668)

On s'aide de la factorisation de $\varphi(n)$ pour trouver un entier $e$ premier avec $\varphi(n)$ :

In [5]:
factor(phi)

2^2 * 23 * 29

On prend $e = 17$. Pour déterminer la clé privée $d$ de déchiffrement telle que $d \cdot e \equiv 1 \pmod{\varphi(n)}$, on utilise l'algorithme d'Euclide étendu avec `xgcd(e, phi)` :

In [6]:
e = 17
xgcd(e, phi)

(1, 157, -1)

On peut donc prendre $d = 157$. On vérifie :

In [7]:
d = 157
(d * e) % phi

1

### 3.2 Chiffrement / Déchiffrement
Le message à chiffrer est `ITS ALL GREEK TO ME.`.
Le premier paquet de 4 chiffres est $M_1 = 0920 = 920$.
Chiffrement avec la clé publique $(n = 2773, e = 17)$ :

In [8]:
c1 = expo_mod(920, e, n)
c1

948

On vérifie le déchiffrement avec la clé privée $(n = 2773, d = 157)$ :

In [9]:
expo_mod(c1, d, n)

920

## 4 Un exemple grandeur nature

### 4.1 Obtenir de grands nombres premiers
On utilise le nombre premier $p$ de 310 chiffres fourni dans le sujet :

In [10]:
p = ZZ(
    "5171580075478931594509360412348407766342210294522059751636118270640506099833200190938948490555"
    "2094870375598598430016472322509555705267350496700893495163926881229099888666976749337348684008"
    "2512338933731996459237216480672992658123322297095622665270892135162788565075722225050864864633"
    "7166668774289018643141435141"
)
len(p.digits())

310

### Exercice 2 : Déterminer un autre nombre premier q de 310 chiffres et valeur de n
On utilise la méthode `ZZ.random_element(10^312)` et `is_pseudoprime()` comme expliqué par le prof :

In [11]:
q = ZZ.random_element(10^312)
while (len(q.digits()) < 310) or (not q.is_pseudoprime()):
    q = ZZ.random_element(10^312)

print("Nombre de chiffres de q :", len(q.digits()))
n = p * q
print("Nombre de chiffres de n :", len(n.digits()))

Nombre de chiffres de q : 311
Nombre de chiffres de n : 621


### 4.2 Calcul de phi(n) (Exercice 3)
Comme expliqué dans le sujet, la commande `euler_phi(n)` prendrait trop de temps car SageMath tenterait de factoriser $n$.
On calcule directement $\varphi(n) = (p - 1)(q - 1)$ :

In [12]:
phi = (p - 1) * (q - 1)
len(phi.digits())

621

### 4.3 & 4.4 Choix des clés et test de chiffrement / déchiffrement

In [13]:
e = 65537
d = xgcd(e, phi)[1] % phi

# Message numérique m au hasard avec m < n :
m = ZZ.random_element(n)

# Chiffrement :
c = expo_mod(m, e, n)

# Déchiffrement et vérification :
expo_mod(c, d, n) == m

True

## 5 Utilisation et attaque du RSA

### 5.1 Algorithme de Fermat (Exercice 4)
Lorsque $p$ et $q$ sont proches, on pose $t = \frac{p+q}{2}$ et $s = \frac{p-q}{2}$, d'où $t^2 - s^2 = n \iff s^2 = t^2 - n$.
On applique l'algorithme de Fermat donné dans l'encadré du sujet :

In [14]:
n = 4840015169768242918240815055699674259180276588222516131662837
e = 65537
c = 2336273333675885101548598149697595180856150539608777837370662

# 1. Factorisation de Fermat
t = floor(sqrt(n)) + 1
z = t^2 - n

while not z.is_square():
    t = t + 1
    z = t^2 - n

p = t + sqrt(z)
q = t - sqrt(z)

p, q

(2200003447671899699253701560601, 2200003447671898699253701560637)

Vérification de la factorisation $p \cdot q == n$ :

In [15]:
p * q == n

True

2. Déduction de la clé privée $d$ et déchiffrement de la clé secrète AES $K$ :

In [16]:
phi = (p - 1) * (q - 1)
d = xgcd(e, phi)[1] % phi
d

3721310471786196994174839096104935750772624293856019894448673

Déchiffrement de la clé secrète $K$ :

In [17]:
K = expo_mod(c, d, n)
K

327957794974300382426813256219279630131

Taille de la clé secrète $K$ (on vérifie qu'elle mesure bien 128 bits comme indiqué dans le sujet) :

In [18]:
K.nbits()

128

### 5.2 Attaque de Wiener (Exercice 5)
On applique exactement la méthode de Wiener présentée dans l'exemple du professeur (page 8 du sujet) avec `continued_fraction(e/n).convergents()` :

In [19]:
mc = 7838294556988410460801911618474336965523762977662313406559253659252836897207399214461066284783654038457202403830044504376648763024360414206299770836089618910088109871939838200354548125816096570413755061613485229318236280200904889067968609369568142098062420460396137162769658214493917660515828417381339578346

n = 72607690796382219533855601574226979839358925894783012264746192242312064761807469686556032798997767218652403849992784150982852102067195784924553481000061931627100261501090420476819158891107478865521004842594565621865761568219736813805495036064204106467048980512337940858538750651798890768604018328115165005573

e = 67715972340573783647161184330298178393906701963991927108027628141059072148144735004419413509389838752160732172897464811380398084788206901478376077107638769067644093024789666701558654700521392247661822616688334643664821383484032774980431792606135434693625403907803225216173220349070640509336914301659248439067

# Calcul des réduites de la fraction continue associée à e/n
cvgts = continued_fraction(e/n).convergents()

# Recherche de la clé privée parmi les dénominateurs des réduites avec le témoin m = 12345
m = 12345
c = power_mod(m, e, n)
i = 0
while i <= len(cvgts) and power_mod(c, cvgts[i].denominator(), n) != m:
    i += 1

d = cvgts[i].denominator()
d

14350959410318879720734159758525562559080808105331415790534726344708163972339

Déchiffrement du cryptogramme $mc$ avec la clé privée $d$ :

In [20]:
m = power_mod(mc, d, n)
m

221110171128141915242415101431101311232631291014152929192415

Décodage du texte français (espace = 10, A = 11, B = 12, ..., Z = 36) :

In [21]:
s = str(m)
texte = ""
for k in range(0, len(s), 2):
    code = int(s[k:k+2])
    if code == 10:
        texte += " "
    elif 11 <= code <= 36:
        texte += chr(ord('A') + (code - 11))
texte

'LA GARDIENNE DU CAMPUS DESSINE'